this note book is the preprocessing for the isochrones approach. 

# 1.0 Load data and packages

In [ ]:
# load the packages
import pandas as pd
import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point
from shapely.wkt import loads
from sklearn.neighbors import NearestNeighbors

In [ ]:
# load the crime data
crime_gdf = gpd.read_file('crime_gdf.geojson', use_arrow=True)

In [ ]:
isochrone_df = gpd.read_file('school_catchments.geojson')#.drop('Unnamed: 0', axis=1)

In [ ]:
# Load cleaned schools
sec_gdf = gpd.read_file('sec_gdf.geojson')

In [ ]:
# Load the previous cleaned bradford 
bradford_boundary = gpd.read_file('bradford_boundary.geojson')

There is no deep data cleaning required here. The datasets have been cleaned from previous buffer approaches.

# 2.0 Data Previewing

## 2.1 Crimes data

In [ ]:
# view crime data
crime_gdf.head(3)

In [ ]:
# check the shape of the data
crime_gdf.shape

In [ ]:
# check data 
crime_gdf.info()

In [ ]:
# confirm null values
crime_gdf.isnull().sum()

In [ ]:
print(crime_gdf.crs)

In [ ]:
# reproject the cordinates
crime_gdf = crime_gdf.to_crs("EPSG:4326")

## 2.2 Boundary

In [ ]:
print(bradford_boundary.crs)

In [ ]:
# convert to the angle
bradford_boundary = bradford_boundary.to_crs("EPSG:4326")

## 2.3 School data

In [ ]:
# check data structure
isochrone_df.shape

In [ ]:
# check teh datatypes
isochrone_df.info()

In [ ]:
# confirm column names
isochrone_df.columns

## 3.0 Data cleaning

## 3.1 Isochrone data

In [ ]:
# Keep only the school names in sec_gdf
sec_cols = sec_gdf['school_name'].unique()
isochrone_df = isochrone_df[isochrone_df['school_name'].isin(sec_cols)]

In [ ]:
isochrone_df.head()

In [ ]:
print(isochrone_df.crs)

I had used three speeds for data extraction, building on pinna & Murau 2018 which found that the speed of children aged 0-18 is 1.0m/s +/- 0.11. I will only apply with the mean speed for this; further work could identify the differences.

In [ ]:
# select only mean speed
isochrone_df = isochrone_df[isochrone_df['walk_speed_ms'] == 1.04]

# 4.0 Sample selection

In [ ]:
# view the crimes with years
crime_gdf['year'].value_counts().to_frame()

In [ ]:
# filter data
crime_gdf = (
    crime_gdf[
    (crime_gdf['year'].isin([2019, 2020])) & 
    (crime_gdf['transition'].isin(['morning commute', 'afternoon commute']))
     ]).copy()

Merge Datasets

In [ ]:
# join the crime data and the isochrone
crime_isochrones = gpd.sjoin(
    crime_gdf, 
    isochrone_df, 
    how='left', 
    predicate='within'
).reset_index()

# 4.0 Exploratory Data Analysis

## 4.1 Overview

In [ ]:
# sort by travel time and take first record for each crime
crime_isochrones = crime_isochrones.sort_values(by='trip_time_min', ascending=True)

# drop duplicates and keep one
crime_isochrones = crime_isochrones.drop_duplicates(subset='index', keep='first')

# set index
crime_isochrones = crime_isochrones.set_index('index')

In [ ]:
crime_isochrones.shape

In [ ]:
# get unique
unique_bands = [5.0, 10.0, 15.0, 20.0, 25.0, 30.0]

In [ ]:
# 4. Loop through each sorted travel bracket to build your isolated maps
for active_band in unique_bands:
    # initialise layout
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # plot boundary
    bradford_boundary.plot(color='none', edgecolor='#666666',
                           linewidth=1.0, linestyle='-', ax=ax)
    
    # plot active time polygon bracket
    isochrone_subset = isochrone_df[isochrone_df['trip_time_min'] <= active_band]
    isochrone_subset.plot(color='#1f77b4', alpha=0,
                          edgecolor='none', linewidth=0, ax=ax)
    
    # plot school location
    isochrone_subset.to_crs("EPSG:27700").centroid.to_crs("EPSG:4326").plot(
        color='black', marker='^', markersize=10, label='School Location', zorder=5, ax=ax)
    
    # get teh active time
    crime_subset = crime_isochrones[crime_isochrones['trip_time_min'] <= active_band]
    
    if not crime_subset.empty:
        crime_subset.plot(color='crimson', markersize=3, alpha=0.5,
                          label='Crime', zorder=4, ax=ax)
    
    # titles and labels
    ax.set_title(f"Crime Incidents Inside the {active_band} Min Walking Isochrone", fontsize=12, fontweight='bold', pad=12)
    ax.axis('off')
    plt.grid(True, linestyle=':', alpha=0.3)
    plt.legend(frameon=True, facecolor='white', edgecolor='none', loc='upper right')
    plt.tight_layout()
    plt.show()

In [ ]:
crime_isochrones.shape

### Save data according to distance 

In [ ]:
# save all the data
df = crime_isochrones.copy()
    
df.to_parquet('crime_isochrones.parquet', index=False)
df.to_file('crime_isochrones.geojson', driver='GeoJSON', index=False)